# What still need to be done
- statistical analysis
- more optimization
- after tweaking hypothesis we need to run  on the posts fetched  from x
- charts
# issues 
- what type of metric we look for
- cma still not that good using text mining
- charts selection 
# feedback
put ur feedback here as bullets points 
- example : not all dataset is needed 

In [60]:
import pandas as pd
from transformers import pipeline , AutoTokenizer , AutoModelForSequenceClassification
import nltk , torch ,re
from nltk.corpus import wordnet, stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import pprint

In [79]:
# this to dowload the dictionary needed by nltk runs ones 
#nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('maxent_ne_chunker_tab')
nltk.download('averaged_perceptron_tagger_eng')

In [63]:
# vars only name  u can change 
name = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name, dtype=torch.float32).eval()
device = "cuda" if torch.cuda.is_available() else "cpu"


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

In [64]:
#help lemitization  the text  
lem = WordNetLemmatizer()

In [65]:
# clensing text from number and pctuation 
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
stop_words = set(stopwords.words('english'))
 
def tokenize_and_filter(text):
    cleaned = clean_text(text)
    tokens = word_tokenize(cleaned.lower())
    return [w for w in tokens if w not in stop_words]
def expand(words):
    out = set(words)
    for w in words:
        for syn in wordnet.synsets(w, pos="n")[:2]:      
            out.update(l.name().lower() for l in syn.lemmas() if "_" not in l.name())
    return out


def to_lemma(tok):
    return lem.lemmatize(lem.lemmatize(tok, "v"), "n")

def frequences(tokens):
    lemmas = [to_lemma(t) for t in tokens]
    return {k: sum(l in words for l in lemmas) for k, words in lexicon.items()}

In [66]:
def score_text(text, hyps, batch_size=8):
    scores = []
    for i in range(0, len(hyps), batch_size):
        batch = hyps[i:i+batch_size]
        inputs = tok([text] * len(batch), batch,
                     return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            logits = model(**inputs).logits          # 3 numbers per pair
        probs = torch.softmax(logits.float(), dim=-1) 
        scores += probs[:, ent_idx].tolist()         # We keep only entailment
    return scores 
# agregation of frames value sum / numner  frame 
# example: 3 respensibility value -> we sum the three probability / 3  
def agr_scores(df , columns):
    norm_scores = pd.DataFrame(columns=columns)
    for c in columns:
        tmp = df[c]
        if (len(tmp.shape) <= 1) :
             norm_scores[c]= tmp    
        else:
            number_frames = tmp.shape[1] 
            norm_scores[c]= tmp.sum(axis=1) / number_frames
    return norm_scores

In [67]:
# it  uses ur gpu or cpu  for torch 
model.to(device)
# We need this later  for calculating 
ent_idx = model.config.label2id["entailment"]

In [68]:
print(model.config.label2id) 
print(model.config.dtype) 

{'contradiction': 2, 'entailment': 0, 'neutral': 1}
torch.float32


In [69]:
# this to test is problem in  the model or our predict function 
premise = "America is SUPPOSED to open her borders to genuine refugees HELLO this is America! We ALL were refugees of one sort or another."
controls = [
    "This text is about a football match.",
    "This text contains a cooking recipe.",
]
concrete = [
    "The text describes refugees as a flood or a wave.",
    "The text describes refugees as an invasion or an attack.",
    "The text describes refugees as vermin or a disease.",
    "The text uses the word 'door', 'wall' or 'house' to talk about borders.",
    "The text says that the government should do something.",
    "The text says that some people disagree about refugees.",
]
print(f"premise :{premise}")
for h in controls + concrete:
    print(round(score_text(premise, [h])[0], 3), h)

premise :America is SUPPOSED to open her borders to genuine refugees HELLO this is America! We ALL were refugees of one sort or another.
0.0 This text is about a football match.
0.0 This text contains a cooking recipe.
0.0 The text describes refugees as a flood or a wave.
0.0 The text describes refugees as an invasion or an attack.
0.0 The text describes refugees as vermin or a disease.
0.529 The text uses the word 'door', 'wall' or 'house' to talk about borders.
0.989 The text says that the government should do something.
0.549 The text says that some people disagree about refugees.


In [70]:
# chage name of file and sheet 's name  if they are diffrent in ur pc
df_hypothesis = pd.read_excel("Human_pilot_llm_pipeline.xlsx",sheet_name="Hypothesis Bank")
df_tweets = pd.read_excel("refugee_tweets_deduplicated.xlsx")

In [71]:
# some vectors used as  columns names 
hypothesis_list = df_hypothesis["hypothesis"].to_list()
ids  = df_hypothesis["id"].tolist()
frames = df_hypothesis["frame / domain"].tolist()
# removed repeated frames
non_rep_frames = list(set(frames))
#hypothesis with no csma  potentially we will used
hypothesis_no_cma= hypothesis_list[0:-5:1]

In [72]:
# seperated per new york times article
ny_ranges=df_tweets[df_tweets["Handle"] == "@nytimes"].index
length_ny_ranges = len(ny_ranges)
tweets_separated_per_article = [
    df_tweets.iloc[ny_ranges[i]:ny_ranges[i + 1]]
    if i + 1 < len(ny_ranges)
    else df_tweets.iloc[ny_ranges[i]:]
    for i in range(0, len(ny_ranges), 1)
]

In [73]:
# resuls of all batch  we need to add id 
all_result = []
for batch_tweets in tweets_separated_per_article:
    rows = [score_text(t, hypothesis_list) for t in batch_tweets['Text']]
    scores_raw = pd.DataFrame(rows, columns=frames).round(3)
    norm_scores = agr_scores(scores_raw,non_rep_frames)
    all_result.append(norm_scores)
result_full_frames = pd.concat(all_result ,  ignore_index=True)   

In [74]:
result_full_frames

,PESTS/DISEASE,Conflict,BUILDINGS/STRUCTURE,Economic Consequences,Morality,WAR,JOURNEY,Responsibility,Human Interest,WATER
0,0.984,0.75475,0.985,0.23400,0.50975,0.994,0.990,0.61850,0.29375,0.984
1,0.984,0.73950,0.963,0.00675,0.93200,0.991,0.987,0.79500,0.68050,0.984
2,0.064,0.30500,0.113,0.00800,0.74725,0.339,0.034,0.02800,0.19500,0.030
3,0.093,0.18675,0.029,0.00250,0.15025,0.096,0.041,0.26725,0.02925,0.035
4,0.980,0.40775,0.226,0.00650,0.23700,0.969,0.886,0.10275,0.07175,0.878
...,...,...,...,...,...,...,...,...,...,...
81,0.957,0.13275,0.137,0.10275,0.53375,0.945,0.055,0.25150,0.54750,0.639
82,0.917,0.91700,0.467,0.01900,0.44125,0.962,0.315,0.04325,0.25150,0.117
83,0.032,0.00300,0.016,0.00475,0.00550,0.056,0.006,0.02350,0.01150,0.012
84,0.007,0.00050,0.003,0.00125,0.00100,0.027,0.002,0.00100,0.00775,0.007


## Hypothesis Tweaking

**Problem:** The original hypotheses gave very high entailment scores (0.96 to 0.99) for almost every item, including metaphor items that were not present in the text.

**Cause:** The hypotheses were long, abstract, and repeated "the refugee issue", so the model matched the topic instead of the specific claim. Controls ("football match", "cooking recipe") scored 0.0, so the model and code are fine.

**Potential Fix:** Rewrite each hypothesis as a short, concrete claim.

In [75]:
# dictionary feel free to add/pop words from it
seeds = {
    "WATER":   ["flood", "wave", "tide", "flow", "stream", "trickle", "drown", "surge"],
    "WAR":     ["invasion", "attack", "battle", "fight", "defense", "siege", "assault"],
    "PESTS":   ["vermin", "infestation", "plague", "swarm", "disease", "contamination"],
    "JOURNEY": ["journey", "path", "route", "destination", "flight", "road"],
    "BUILD":   ["wall", "door", "house", "foundation", "gate", "fence"],
}
for  s in seeds.keys():
    res = expand(seeds[s])
    seeds[s] = res
print("seed after expanding")
pprint.pprint(seeds)

seed after expanding
{'BUILD': {'door',
           'doorway',
           'fence',
           'fencing',
           'firm',
           'foundation',
           'gate',
           'house',
           'threshold',
           'wall'},
 'JOURNEY': {'destination',
             'finish',
             'flight',
             'flying',
             'goal',
             'itinerary',
             'journey',
             'journeying',
             'path',
             'road',
             'route',
             'terminus',
             'way'},
 'PESTS': {'cloud',
           'contaminant',
           'contamination',
           'disease',
           'drove',
           'horde',
           'infestation',
           'pest',
           'pestilence',
           'pestis',
           'plague',
           'swarm',
           'taint',
           'varmint',
           'vermin'},
 'WAR': {'assault',
         'attack',
         'battle',
         'beleaguering',
         'besieging',
         'combat',
        

In [76]:
result_after_tok = []
for batch_tweets in tweets_separated_per_article:
    raw_texts = batch_tweets["Text"].tolist()
    result_after_tok.extend(tokenize_and_filter(t) for t in raw_texts)
print(len(result_after_tok)) 

86


In [77]:
# keep the originals, build the expanded lexicon separately
lexicon = {k: expand(v) for k, v in seeds.items()}
# always print and read the lists
p = [frequences(toks) for toks in result_after_tok]
cma_counts = pd.DataFrame(p)

In [78]:
# total hits per domain
print(cma_counts.sum())

hits = cma_counts[cma_counts.sum(axis=1) > 0]
print(len(hits), "of", len(cma_counts), "posts have at least one hit")

# show the actual posts that matched
all_posts = pd.concat(tweets_separated_per_article, ignore_index=True)
for i in hits.index:
    print(i, hits.loc[i][hits.loc[i] > 0].to_dict(), "|", all_posts.loc[i, "Text"][:100])

WATER      3
WAR        1
PESTS      0
JOURNEY    7
BUILD      7
dtype: int64
14 of 86 posts have at least one hit
20 {'BUILD': 1} | ok. We'll send them to your house. What is that address again?
25 {'WAR': 1} | UNLIKE OUR #PresidentObama WE ARE NOT WEAK AND WILL STAND AND FIGHT FOR OUR COUNTRY!!!!!
29 {'JOURNEY': 1} | way to stir up the lunkheads.
34 {'BUILD': 1} | Publish their home addresses too you morons. Go put signs on their doors too. Idiots.
39 {'JOURNEY': 1} | way to go NYTimes, put a target on innocent people. Why not list addresses, phone numbers & pictures
52 {'BUILD': 2} | how come USA will build walls for Mexicans but not walls for terrorist? Potential and future terrori
56 {'WATER': 1} | ARTICLE: The first Ukrainian refugees are trickling into the United States through informal avenues 
59 {'JOURNEY': 1} | Biden was escalating this war so it is his responsibility to care for refugees from Ukraine!! Direct
60 {'JOURNEY': 2} | United States is the destination of centuries